# Polars: detailed scenarios

Expressions, group_by, windows, and lazy execution. Needs the **polars** package (in the lab dependencies). Self-contained. See **polars_cheatsheet.md**.

In [ ]:
import numpy as np
import polars as pl
import datetime as dt
rng = np.random.default_rng(2)

n = 180
dates = [dt.date(2023, 1, 2) + dt.timedelta(days=i) for i in range(n)]
frames = []
for name, p0, mu, sig in [("ACME", 98, 0.0004, 0.014), ("GLOBEX", 51, 0.0002, 0.010), ("INITECH", 210, 0.0009, 0.020)]:
    close = p0 * np.cumprod(1 + rng.normal(mu, sig, n))
    frames.append(pl.DataFrame({"date": dates, "ticker": [name] * n, "close": close}))
df = pl.concat(frames)
df.head()

## Expressions: select, with_columns, filter

An expression describes a computation on columns; the engine runs it.

In [ ]:
out = (
    df.sort(["ticker", "date"])
      .with_columns(pl.col("close").pct_change().over("ticker").alias("ret"))
      .filter(pl.col("ret").is_not_null())
)
print(out.select(pl.col("ret").mean().alias("mean"), pl.col("ret").std().alias("vol")))

## Group by: mean and volatility per ticker

In [ ]:
summary = (
    df.sort(["ticker", "date"])
      .with_columns(pl.col("close").pct_change().over("ticker").alias("ret"))
      .group_by("ticker")
      .agg(pl.col("ret").mean().alias("mean"), pl.col("ret").std().alias("vol"))
      .sort("vol")
)
print(summary)

## Time grouping: monthly mean return

Truncate the date to the month, then group.

In [ ]:
monthly = (
    df.sort(["ticker", "date"])
      .with_columns(
          pl.col("close").pct_change().over("ticker").alias("ret"),
          pl.col("date").dt.truncate("1mo").alias("month"),
      )
      .group_by(["ticker", "month"])
      .agg(pl.col("ret").mean().alias("mean_ret"))
      .sort(["ticker", "month"])
)
print(monthly.head(6))

## Window functions: over() and rolling

In [ ]:
w = df.filter(pl.col("ticker") == "ACME").with_columns(
    pl.col("close").pct_change().alias("ret"),
    pl.col("close").rolling_mean(20).alias("sma20"),
)
print(w.tail(3))

## Lazy execution: build a plan, then collect

In [ ]:
lf = (
    df.lazy()
      .with_columns(pl.col("close").pct_change().over("ticker").alias("ret"))
      .group_by("ticker")
      .agg(pl.col("ret").mean().alias("mean_ret"))
)
print(lf.collect().sort("ticker"))

## Recap

Expressions, group_by, monthly grouping, windows with over and rolling, and lazy collect.